# NLG LLM classification

Runs our prompt on every complaint narrative through OpenRouter, checks every answer, and saves the results.

The settings in cell 1 are the ones used for the full run: Claude Sonnet 5.5, 15 cases per call, prompt version `402f60b2`. To repeat the earlier model test, set `CASES_FILE` to `sample_complaints_llm.csv`, `MODELS` to Haiku, Sonnet and Opus, and `BATCH_SIZES` to `[1, 15]`.

This notebook lives in `LLM Classification/` and reads the data from `Joining Datasets/Data/data for llm/`.

The notebook asks for the API key in a hidden box. **Never paste the key into a cell.** Clear all outputs before committing.

## 1. Settings

In [ ]:
%pip install -q requests pandas

import os, re, json, time, getpass, difflib, hashlib
from collections import Counter
import pandas as pd
import requests

DATA_DIR = "../Joining Datasets/Data/data for llm"  # folder with the LLM files from joining.ipynb
CASES_FILE = "llm_final_df.csv"                     # model test used "sample_complaints_llm.csv"
MODELS = ["anthropic/claude-sonnet-5.5"]            # model test also ran claude-haiku-5.5 and claude-opus-5.5
BATCH_SIZES = [15]                                  # cases per call. Model test also ran 1
FULL_RUN_CASES = 3086                               # cases the full run will send, used for cost estimates
OUT_DIR = "llm_results"
SEED = 42                                           # keeps the shuffle the same every time

API_KEY = os.environ.get("OPENROUTER_API_KEY") or getpass.getpass("Paste the OpenRouter API key, then press Enter: ")
os.makedirs(OUT_DIR, exist_ok=True)
print("Settings loaded. Key received:", bool(API_KEY))

## 2. The prompt

In [ ]:
PROMPT = r"""
You are a neutral, impartial coder for a research project on Boston Police complaint narratives. You see only each case's ID and narrative, never BPD's labels or outcomes. Code only what the narrative alleges. Code each case on its own and do not compare cases.

For each case return:
- id
- alg: 1 to 3 allegation codes from ALLEGATIONS that fit the alleged conduct, most central first.
- sev: one SEVERITY code for the most serious conduct alleged, or "UNCLEAR".
- conf: 1 to 5, how clearly the text supports sev. 1 = guess, 2 = unsure, 3 = moderate, 4 = confident, 5 = the text states the conduct directly.
- lang: any that apply, else []. P = procedural or form-like wording. V = vague or minimized, leaves out what actually happened. G = garbled, cut off, or repeated text.
- quote: up to 12 words copied exactly from the narrative, typos included, that best support sev.

ALLEGATIONS
General conduct
1 Neglect of Duty / Unreasonable Judgment
2 Respectful Treatment
3 Conduct Unbecoming
4 Conformance to Laws (officer broke a law)
5 Self Identification (refused name, badge, or ID)
6 Directives and Orders
7 Abuse of Process (misused legal process, e.g. baseless charge or citation)
8 Abuse of Discretion
9 General Conduct / Rule Violations (only if nothing more specific fits)
10 Police Service (failed to help or respond)
11 Ethics Canons
Discrimination and harassment
12 Bias / Discriminatory Treatment
13 Harassment (incl. sexual harassment, retaliation for reporting)
14 Hostile Work Environment
15 Conflicts / Off-Duty Situations (family, friends, off-duty officers)
16 Association with Criminals
Force, searches, custody
17 Use of Force
18 Search / Handcuff Procedures
19 Strip Search
20 Prisoner Care
21 Booking Procedures
22 Officer Safety / Mutual Protection
Truthfulness and records
23 Untruthfulness / False Reporting
24 Inaccurate Reporting
25 Reporting / Documentation
26 Complaint Intake Procedures
27 CJIS Access
28 CORI Access
29 Information Disclosure
30 Statements / Public Communications
31 Interview Procedures
32 Criminal Complaint Procedures
Attendance and supervision
33 Paid Details
34 Reporting for Duty
35 Duty / Supervisor Notification
36 Supervisory Responsibilities
37 Leave / Attendance
38 Residence / Address Reporting
39 Personal Business
40 Sleeping on Duty
41 Court Procedures
Alcohol and drugs
42 Alcohol on Duty
43 Alcohol in Police Facility
44 Alcohol off Duty
45 Drug Testing
46 Substance Abuse
47 Controlled Substances
48 Substance / Fitness for Duty
49 Tobacco on Duty
Vehicles and equipment
50 Uniform and Equipment
51 Body-Worn Camera
52 Emergency Driving
53 Department Vehicle
54 Evidence / Property
55 Firearms
56 Communications / Radio
57 Computer / Email Use

SEVERITY
UNCLEAR: no officer conduct is described at all
Level 1 Administrative
1.1 Paperwork or notification: missing reports or FIOs, not notifying a supervisor, not giving name or badge
1.2 Attendance, duty, paid details: late, absent, sick time misuse, left post, sleeping, unapproved outside work
1.3 Equipment or procedure: body camera, uniform, firearm storage, not following orders
1.4 Drugs or alcohol: failed drug test, under the influence, alcohol on or off duty
1.5 Dishonesty or misuse of access: untruthful, false detail or time cards, CJIS or CORI misuse, mishandled evidence or money
Level 2 Non-physical misconduct
2.1 Disrespect or intimidation: rude, yelling, cursing, mocking, hostile, no threat
2.2 Neglect or failure to act: refused to help, did not respond, investigate, or intervene
2.3 Threat or coercion: explicit or implied threats, threat of arrest or ticket, retaliation, using authority to pressure
2.4 Harassment, discrimination, prejudice: repeated harassment, slurs, profiling, targeting by race or other identity, sexual harassment, improper search with no body contact
Level 3 Physical contact or force
3.1 Minor contact or control: grab, push, pull, handcuff, pat-down, takedown during arrest, no injury needing care, not described as unnecessary
3.2 Unjustified force, any one of: force called unnecessary or excessive; strikes such as punch, kick, slam; injury needing medical care; weapon or less-lethal tool used, incl. gun pointed, pepper spray, baton, taser; sexual or intrusive contact, incl. strip search
3.3 Deadly force or severe harm: shooting, choking, unconsciousness, not breathing, life-threatening injury, torture

RULES
1. Score the conduct as alleged. Do not raise or lower it because it is proven, unproven, or denied. Ignore any findings, discipline, or recommendations in the text. Do not infer facts that are not in the text.
2. Severity comes from the described conduct, not from the allegation code you chose.
3. Do not judge credibility. Who wrote the narrative (complainant, witness, attorney, IAD, anonymous caller) must not change alg or sev. Neither may the race, gender, age, disability, mental health, criminal record, or custody status of anyone involved, or the narrative's length, grammar, spelling, or emotional tone. conf reflects only how clearly the conduct is described.
4. Describing someone's race or other identity is not bias by itself. Code bias when the narrative says or shows someone was treated differently because of identity, including when the complainant says they believe that is why.
5. Use only the narrative. Ignore anything you may know about the officers, people, or events.
6. If more than one severity applies, pick the highest level, then the higher sublevel.
7. Physical contact that the narrative complains about is Level 3, whatever the allegation code. Handcuffing, detaining, or searching a person counts when the narrative says the stop, arrest, or search was unjustified, rough, or illegal. It is background only when the narrative complains about something else.
8. Searches: car, home, or belongings with no body contact = 2.4. Pat-down or frisk = 3.1. Strip or body cavity search = 3.2.
9. Score what the officer did toward anyone, on or off duty, including coworkers and family. An officer being arrested or charged counts as described conduct, so code the alleged act.
10. The text may have OCR errors, missing letters, and [NAME] placeholders. Read through them.
11. Do not default to a low severity when the text is unclear. Pick the closest code and lower conf instead. Use UNCLEAR only if no officer conduct is described, and then alg is [].

OUTPUT
Respond with a JSON array only, no reasoning, preamble, or markdown. One object per case, in the order given. Do not repeat the narrative.
[{"id":"IAD2000-0000","alg":[2,7],"sev":"2.3","conf":3,"lang":["P"],"quote":"exact words from the narrative"}]
"""

SYSTEM_PROMPT = PROMPT.strip()
PROMPT_VERSION = hashlib.md5(SYSTEM_PROMPT.encode()).hexdigest()[:8]

def section(name, next_name):   # text between two heading lines, e.g. ALLEGATIONS and SEVERITY
    after = re.split(rf"^\s*{name}\s*$", SYSTEM_PROMPT, flags=re.M)[1]
    return re.split(rf"^\s*{next_name}\s*$", after, flags=re.M)[0]

# Read the allegation names and severity codes from the prompt, so the checks always match it
ALG_NAMES = {int(n): re.sub(r"\s*\(.*\)\s*$", "", name).strip()
             for n, name in re.findall(r"^(\d+)\s+(.+)$", section("ALLEGATIONS", "SEVERITY"), flags=re.M)}
SEV_CODES = set(re.findall(r"^(\d\.\d)\s", section("SEVERITY", "RULES"), flags=re.M)) | {"UNCLEAR"}
print(f"Prompt version {PROMPT_VERSION}: {len(ALG_NAMES)} allegation codes, {len(SEV_CODES)} severity codes")

## 3. Load and clean the cases

In [ ]:
path = os.path.join(DATA_DIR, CASES_FILE)
if not os.path.exists(path):
    raise FileNotFoundError(f"Can't find {path}. Change DATA_DIR in cell 1 to the folder that has {CASES_FILE}.")
cases = pd.read_csv(path).rename(columns={"narrative_x": "narrative"})
cases["ia_no"] = cases["ia_no"].astype(str)
cases["narrative"] = cases["narrative"].fillna("").astype(str)

# Some narratives are the same text pasted twice with " | ". Keep one copy, the cleaner one.
word_counts = Counter(w.lower() for t in cases["narrative"] for w in re.findall(r"[A-Za-z]+", t))
def junk_score(text):   # share of words that appear nowhere else, which is high for OCR garble
    words = re.findall(r"[A-Za-z]+", text)
    return sum(word_counts[w.lower()] == 1 for w in words) / max(len(words), 1)

def dedupe(text):
    keep = []
    for part in [p.strip() for p in text.split(" | ") if p.strip()]:
        for i, kept in enumerate(keep):
            same_start = difflib.SequenceMatcher(None, part[:600].lower(), kept[:600].lower()).ratio()
            same_end = difflib.SequenceMatcher(None, part[-600:].lower(), kept[-600:].lower()).ratio()
            if max(same_start, same_end) > 0.85:
                if junk_score(part) < junk_score(kept):
                    keep[i] = part
                break
        else:
            keep.append(part)
    return " | ".join(keep)

cases["narrative"] = cases["narrative"].map(dedupe)
cases["too_short"] = cases["narrative"].str.count(r"[A-Za-z]{3,}") < 5   # nothing to code, not sent
cases = cases.sample(frac=1, random_state=SEED).reset_index(drop=True)   # shuffle
NARRATIVES = dict(zip(cases["ia_no"], cases["narrative"]))
print(f"{len(cases)} cases loaded, {cases['too_short'].sum()} too short to send")

## 4. Helper functions

In [ ]:
URL = "https://openrouter.ai/api/v1/chat/completions"

def build_user_message(batch):
    lines = [f'<case id="{r.ia_no}">{r.narrative}</case>' for r in batch.itertuples()]
    return "COMPLAINTS\n" + "\n".join(lines)

def call_model(model, batch, effort="none"):
    """Send one batch of cases. Returns (output text, usage, reasoning effort used)."""
    body = {
        "model": model,
        "temperature": 0,
        "max_tokens": 400 + 110 * len(batch),
        "reasoning": {"effort": effort},
        "messages": [{"role": "system", "content": SYSTEM_PROMPT},
                     {"role": "user", "content": build_user_message(batch)}],
    }
    for attempt in range(8):
        r = requests.post(URL, headers={"Authorization": f"Bearer {API_KEY}"}, json=body, timeout=300)
        if r.status_code == 400 and body["reasoning"]["effort"] == "none" and "reason" in r.text.lower():
            body["reasoning"]["effort"] = "low"          # this model won't turn reasoning off
            continue
        if r.status_code == 402 and "in_flight" in r.text:   # too much spend in flight at once, OpenRouter says wait
            try: wait = int(float(r.headers.get("Retry-After") or 120))
            except ValueError: wait = 120
            print(f"  OpenRouter asked us to wait {wait}s, waiting...")
            time.sleep(wait)
            continue
        if r.status_code in (429, 500, 502, 503, 504):   # busy or rate limited, wait and retry
            time.sleep(5 * (attempt + 1))
            continue
        if r.status_code != 200:
            raise RuntimeError(f"{model} returned error {r.status_code}: {r.text[:500]}")
        data = r.json()
        return data["choices"][0]["message"]["content"] or "", data.get("usage", {}), body["reasoning"]["effort"]
    raise RuntimeError(f"{model} kept failing after retries")

def repair(t):
    t = re.sub(r'"\s*,\s*"\s*\}', '"}', t)       # Sonnet sometimes writes ...","} at the end of an answer
    t = re.sub(r',\s*([}\]])', r'\1', t)          # trailing commas
    return t

def parse_output(text):
    t = repair(re.sub(r"^```(?:json)?|```$", "", text.strip(), flags=re.M).strip())
    try:
        out = json.loads(t[t.find("["): t.rfind("]") + 1])
    except Exception:
        out = []                                  # read each answer on its own, so one bad one doesn't sink the batch
        starts = [m.start() for m in re.finditer(r'\{\s*"id"', t)]
        for i, s in enumerate(starts):
            piece = t[s: starts[i + 1] if i + 1 < len(starts) else len(t)].strip().rstrip("],").strip()
            try: out.append(json.loads(repair(piece)))
            except Exception: pass
    return [o for o in out if isinstance(o, dict)]

def norm(s):
    s = str(s).lower().strip().strip('"').strip("'").replace("...", " ").replace("…", " ")
    s = re.sub(r"[‐-―−]", "-", s)
    s = re.sub(r"[“”]", '"', s)
    s = re.sub(r"[‘’]", "'", s)
    return re.sub(r"\s+", " ", s).strip()

def check_row(o, narrative):
    problems = []
    if str(o.get("sev")) not in SEV_CODES:
        problems.append("bad sev code")
    algs = o.get("alg") or []
    if not isinstance(algs, list) or any(a not in ALG_NAMES for a in algs):
        problems.append("bad alg code")
    if str(o.get("sev")) != "UNCLEAR" and not algs:
        problems.append("no alg")
    if not isinstance(o.get("conf"), int) or not 1 <= o.get("conf") <= 5:
        problems.append("bad conf")
    q = norm(o.get("quote", ""))
    if q and q not in norm(narrative):
        problems.append("quote not in narrative")
    return problems

def raw_path(model, batch_size):
    return os.path.join(OUT_DIR, f"raw_{model.replace('/', '_')}_batch{batch_size}_prompt{PROMPT_VERSION}.jsonl")

def read_raw(model, batch_size):
    p = raw_path(model, batch_size)
    return [json.loads(line) for line in open(p, encoding="utf-8")] if os.path.exists(p) else []

def run(model, batch_size):
    """Send every case not already done. Safe to rerun: it picks up where it stopped."""
    done = {str(o.get("id")) for rec in read_raw(model, batch_size) for o in parse_output(rec["output"])}
    todo = cases[~cases["too_short"] & ~cases["ia_no"].isin(done)]
    batches = [todo.iloc[i:i + batch_size] for i in range(0, len(todo), batch_size)]
    sent, effort = 0, None
    with open(raw_path(model, batch_size), "a", encoding="utf-8") as raw:
        while batches:
            batch = batches.pop(0)
            text, usage, effort = call_model(model, batch)
            raw.write(json.dumps({"ids": list(batch["ia_no"]), "output": text, "usage": usage,
                                  "effort": effort, "time": time.strftime("%Y-%m-%d %H:%M")}) + "\n")
            raw.flush()
            sent += 1
            got = {str(o.get("id")) for o in parse_output(text)}
            missing = [i for i in batch["ia_no"] if i not in got]
            if missing and len(batch) > 1:            # rerun any case the model skipped, one at a time
                batches = [batch[batch["ia_no"] == i] for i in missing] + batches
            if sent % 10 == 0:
                print(f"  {sent} calls sent, {len(batches)} batches left")
    print(f"{model} x{batch_size}: {sent} calls sent ({len(done)} cases were already done). Reasoning effort: {effort}")

def results_for(model, batch_size):
    """Turn the saved raw output into one row per case."""
    recs = read_raw(model, batch_size)
    latest = {}
    for rec in recs:
        for o in parse_output(rec["output"]):
            latest[str(o.get("id"))] = o
    rows = []
    for r in cases.itertuples():
        o = latest.get(r.ia_no)
        if r.too_short:
            rows.append({"ia_no": r.ia_no, "alg": [], "alg_names": "", "sev": "UNCLEAR", "conf": None,
                         "lang": "G", "quote": "", "problems": "too short, not sent"})
        elif o is None:
            rows.append({"ia_no": r.ia_no, "problems": "no output"})
        else:
            algs = o.get("alg") or []
            rows.append({"ia_no": r.ia_no, "alg": algs,
                         "alg_names": "|".join(ALG_NAMES.get(a, f"?{a}") for a in algs) if isinstance(algs, list) else "",
                         "sev": str(o.get("sev")), "conf": o.get("conf"), "lang": "|".join(o.get("lang") or []),
                         "quote": o.get("quote", ""), "problems": "; ".join(check_row(o, r.narrative))})
    df = pd.DataFrame(rows)
    df.insert(1, "run", f"{model.split('/')[-1]} x{batch_size}")
    df["prompt_version"] = PROMPT_VERSION
    cost = sum(float((rec.get("usage") or {}).get("cost") or 0) for rec in recs)
    return df, cost

## 5. Quick check with one case

Run this first. If it prints a result and a cost, the key works.

In [ ]:
test_text, test_usage, test_effort = call_model(MODELS[0], cases[~cases["too_short"]].head(1))
print(test_text)
print("cost: $", test_usage.get("cost"), "| tokens in/out:", test_usage.get("prompt_tokens"), "/",
      test_usage.get("completion_tokens"), "| reasoning tokens:",
      (test_usage.get("completion_tokens_details") or {}).get("reasoning_tokens"))

## 6. Run

If this stops partway (internet drops, laptop sleeps), just run it again. It skips the cases that are already done.

In [ ]:
all_results, costs = [], {}
for model in MODELS:
    for bs in BATCH_SIZES:
        run(model, bs)
        df, cost = results_for(model, bs)
        all_results.append(df)
        costs[df["run"].iloc[0]] = cost

results = pd.concat(all_results, ignore_index=True)
out_file = os.path.join(OUT_DIR, f"results_{CASES_FILE.replace('.csv', '')}_prompt{PROMPT_VERSION}.csv")
results.to_csv(out_file, index=False)
for name, cost in costs.items():
    n_prob = (results.loc[results["run"] == name, "problems"] != "").sum()
    print(f"{name}: cost ${cost:.4f}, {n_prob} cases with problems")
print(f"\nTotal cost: ${sum(costs.values()):.4f}. Saved to {out_file}")

## 7. Costs

What each run actually cost on OpenRouter, and what the full run would cost at the same rate.

In [ ]:
cost_rows = []
for model in MODELS:
    for bs in BATCH_SIZES:
        recs = read_raw(model, bs)
        usage = [rec.get("usage") or {} for rec in recs]
        cases_done = (results["run"] == f"{model.split('/')[-1]} x{bs}").sum() - cases["too_short"].sum()
        cost = sum(float(u.get("cost") or 0) for u in usage)
        per_case = cost / max(cases_done, 1)
        cost_rows.append({
            "model": model.split("/")[-1],
            "cases per call": bs,
            "calls": len(recs),
            "cases coded": cases_done,
            "input tokens": sum(int(u.get("prompt_tokens") or 0) for u in usage),
            "output tokens": sum(int(u.get("completion_tokens") or 0) for u in usage),
            "thinking tokens": sum(int((u.get("completion_tokens_details") or {}).get("reasoning_tokens") or 0) for u in usage),
            "test cost ($)": round(cost, 4),
            "cost per case ($)": round(per_case, 5),
            f"full run estimate, {FULL_RUN_CASES} cases ($)": round(per_case * FULL_RUN_CASES, 2),
        })
cost_table = pd.DataFrame(cost_rows)
cost_table.to_csv(os.path.join(OUT_DIR, f"costs_prompt{PROMPT_VERSION}.csv"), index=False)
display(cost_table)

## 8. Compare the runs

In [ ]:
sev = results.pivot_table(index="ia_no", columns="run", values="sev", aggfunc="first")
display(sev)

def level(x):
    return str(x).split(".")[0] if str(x) not in ("None", "nan", "UNCLEAR") else "UNCLEAR"

runs = list(sev.columns)
pairs = []
for i in range(len(runs)):
    for j in range(i + 1, len(runs)):
        a, b = sev[runs[i]], sev[runs[j]]
        pairs.append({"run A": runs[i], "run B": runs[j],
                      "same sublevel": f"{(a == b).mean():.0%}",
                      "same level": f"{(a.map(level) == b.map(level)).mean():.0%}"})
print("How often the runs agree with each other:")
display(pd.DataFrame(pairs))

print("Cases with problems (want this near 0):")
display(results[results["problems"] != ""][["run", "ia_no", "problems"]])

## 10. How much the API key has spent and what's left

In [ ]:
h = {"Authorization": f"Bearer {API_KEY}"}
k = requests.get("https://openrouter.ai/api/v1/key", headers=h).json()["data"]
print(f"Spent on this key, all time: ${k['usage']:.2f} (today: ${k.get('usage_daily') or 0:.2f})")
if k.get("limit") is not None:
    print(f"Key limit: ${k['limit']:.2f} | Left on this key: ${k['limit_remaining']:.2f}")
else:
    try:
        d = requests.get("https://openrouter.ai/api/v1/credits", headers=h).json()["data"]
        print(f"This key has no limit. Account credits left: ${d['total_credits'] - d['total_usage']:.2f}")
    except Exception:
        print("This key has no limit. The account balance has to be checked by the key owner at openrouter.ai/settings/credits")